# Track 2 · 01 Dreyer 2023 prepare — the warm-up data onto Drive, as the harness makes it

**One job.** Put the warm-up corpus on Drive in the form `track2_02_eegnet` trains on: the
harness's own 4-second windows, their left/right labels, and the people the warm-up tests.

Self-contained: upload this one file.

---

## The task, from source

The warm-up scores on **Dreyer 2023** (`tracks/bci_decoding/README.md`: "the warm-up
evaluation study, default"). 87 people imagining left- or right-hand movement, 27 EEG
channels at 512 Hz, 6 runs of 40 cued trials each (one person has 4 runs). Metric:
balanced accuracy, so chance is 50%.

From `neuralbench/tasks/eeg/motor_imagery/config.yaml` and its `dreyer2023.yaml`
overlay: one window per cue, **0 to 4 s after the cue**, labelled left (0) or right (1).
**The test people are 61–81** (part B of the dataset), fixed by the organisers so that
train and test people never overlap. They must not be trained on.

The data is read exactly as the harness reads it: through **MOABB's `Dreyer2023`**
loader, which `neuralfetch` wraps.

## Preprocessing is the harness's, exactly

| step | value | at 512 Hz |
|---|---|---|
| notch | 50 and 60 Hz plus harmonics below Nyquist | 50, 100, 150, 200, 250 and 60, 120, 180, 240 |
| band-pass | 0.1 – 75 Hz | both edges apply |
| resample | 120 Hz | 480 samples per window |
| scale | sklearn `RobustScaler` per channel over the whole run | |
| baseline | none for this task | |
| clamp | ±20 | |

Checked against neuralset's own extractor on a real run: identical windows (max
difference 0.0), cue onsets between samples included.

## Licence

Dreyer 2023 is **CC-BY-4.0** (the licence field in neuralfetch's study file). Cite
Dreyer et al., *Scientific Data* 2023.

## Cost

About 6.3 GB downloaded from OSF, one person at a time to temporary disk and deleted
after. About 540 MB written to Drive (float16; rounding error under 0.003 against a
signal sd of about 0.8). Roughly 30–45 minutes. It resumes: people already on Drive are
skipped on a rerun.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

# The harness's settings. Do not tune these; matching them is the point.
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
CLAMP = 20.0
START = 0.0          # window start relative to the cue, s
DURATION = 4.0       # window length, s

SUBJECTS = list(range(1, 88))
TEST_SUBJECTS = list(range(61, 82))     # dreyer2023.yaml PredefinedSplit
EVENT_CODES = {'left_hand': 1, 'right_hand': 2}   # neuralfetch event_id; label = sorted index

print('NOTEBOOK VERSION: track2_01 v1')

In [ ]:
%pip install -q 'moabb>=1.7.1' mne

import importlib
for mod in ('moabb', 'mne', 'sklearn', 'numpy'):
    print(f'{mod:8} {importlib.import_module(mod).__version__}')

In [ ]:
import json, logging, os, shutil, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, mne
from sklearn.preprocessing import RobustScaler
mne.set_log_level('ERROR')
logging.getLogger('moabb').setLevel(logging.ERROR)

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

OUT = Path(ROOT) / 'Track2' / 'dreyer2023'
WIN = OUT / 'windows'
WIN.mkdir(parents=True, exist_ok=True)
TMP = Path('/content/moabb_tmp') if Path('/content').exists() else OUT.parent / '_moabb_tmp'
TMP.mkdir(parents=True, exist_ok=True)
mne.set_config('MNE_DATA', str(TMP), set_env=True)    # MOABB downloads go to temporary disk

from moabb.datasets import Dreyer2023
LABEL = {code: i for i, code in enumerate(sorted(EVENT_CODES.values()))}   # 1 -> 0, 2 -> 1

## Step 1 — download, preprocess and window each person

Each person is fetched through MOABB, every run is preprocessed on its own (the scaler is
fit per run, as the harness does), windows are cut from each cue, and the person's files
are deleted from temporary disk before the next one.

In [ ]:
def mi_windows(raw, onsets):
    """Harness default preprocessing for one run, then 4-s windows from each cue."""
    raw = raw.copy().pick('eeg').load_data()
    nyq = raw.info['sfreq'] / 2
    freqs = [f for base in NOTCH for f in np.arange(base, min(nyq, 301), base)]
    if freqs:
        raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)
    raw.resample(SFREQ, verbose=False)
    d = RobustScaler().fit_transform(raw.get_data().T).T
    n_times = int(round(DURATION * SFREQ))
    idx = np.round((np.asarray(onsets) + START) * SFREQ).astype(int)
    keep = (idx >= 0) & (idx + n_times <= d.shape[1])
    X = np.stack([d[:, i:i + n_times] for i in idx[keep]]).astype('float32')
    return np.clip(X, -CLAMP, CLAMP), keep, list(raw.ch_names)


def do_subject(s):
    dest = WIN / f'sub-{s:02d}.npz'
    if dest.exists():
        return 'kept', json.loads(str(np.load(dest)['info']))
    runs = Dreyer2023().get_data(subjects=[s])[s]
    Xs, ys, rs, ch_ref, per_run = [], [], [], None, {}
    for sess in runs.values():
        for name, raw in sess.items():
            a = raw.annotations
            m = np.array([d in EVENT_CODES for d in a.description])
            onsets = a.onset[m]
            labels = np.array([LABEL[EVENT_CODES[d]] for d in np.array(a.description)[m]])
            X, keep, ch = mi_windows(raw, onsets)
            if ch_ref is None:
                ch_ref = ch
            elif ch != ch_ref:
                raise RuntimeError(f'run {name}: channels differ from the first run')
            Xs.append(X.astype('float16')); ys.append(labels[keep])
            rs.append(np.full(int(keep.sum()), name)); per_run[name] = int(keep.sum())
    info = dict(subject=s, channels=ch_ref, runs=per_run, n_windows=int(sum(per_run.values())),
                native_sfreq=512.0)
    np.savez(dest, X=np.concatenate(Xs), y=np.concatenate(ys).astype('int64'),
             run=np.concatenate(rs), info=json.dumps(info))
    for p in list(TMP.rglob(f'sub-{s:02d}*')):   # this person's zip and extracted files
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
    return 'new', info


t0 = time.time()
done, failed = {}, {}
for i, s in enumerate(SUBJECTS, 1):
    try:
        status, info = do_subject(s)
        done[s] = info
        print(f'  {i:2d}/{len(SUBJECTS)}  sub-{s:02d} {status}  {info["n_windows"]} windows  '
              f'{time.time() - t0:.0f}s')
    except Exception as e:
        failed[s] = repr(e)
        print(f'  FAILED sub-{s:02d}: {e}')
print(f'\n{len(done)} people processed, {len(failed)} failed, {time.time() - t0:.0f}s')
if failed:
    print('Rerun this cell: finished people are kept and only the failures are retried.')

## Step 2 — groups, manifest and checks

`test` is the organisers' 61–81. `val` is ours: 20% of the remaining people, seed 33, held
out whole for choosing epochs. The harness draws its own validation people internally,
which only affects its training runs, not the score.

In [ ]:
from sklearn.model_selection import train_test_split

ALL = list(range(1, 88))                  # the whole study, whatever was processed above
pool = [s for s in ALL if s not in TEST_SUBJECTS]
_, val = train_test_split(pool, test_size=0.2, random_state=33, shuffle=True)
group = {s: 'test' if s in TEST_SUBJECTS else 'val' if s in val else 'train' for s in ALL}

per_group, problems, recs, chans = {}, [], {}, set()
for f in sorted(WIN.glob('sub-*.npz')):
    z = np.load(f)
    info = json.loads(str(z['info']))
    X, y = z['X'], z['y']
    g = group[info['subject']]
    recs[f.stem] = dict(info, group=g, file=f.name)
    chans.add(tuple(info['channels']))
    if X.shape[1:] != (27, int(DURATION * SFREQ)):
        problems.append(f'{f.name}: shape {X.shape}')
    if not (np.isfinite(X.astype('float32')).all() and np.abs(X).max() <= CLAMP):
        problems.append(f'{f.name}: values out of range')
    if set(np.unique(y)) - {0, 1}:
        problems.append(f'{f.name}: labels {np.unique(y)}')
    p = per_group.setdefault(g, dict(people=0, runs=0, windows=0, left=0, right=0))
    p['people'] += 1; p['runs'] += len(info['runs']); p['windows'] += len(y)
    p['left'] += int((y == 0).sum()); p['right'] += int((y == 1).sum())

for g in ('train', 'val', 'test'):
    p = per_group.get(g, {})
    print(f'  {g:5}  {p.get("people", 0):2d} people  {p.get("runs", 0):3d} runs  '
          f'{p.get("windows", 0):6,d} windows   left {p.get("left", 0):,} / right {p.get("right", 0):,}')
if len(chans) != 1:
    problems.append(f'{len(chans)} different channel lists across people')
total_runs = sum(p['runs'] for p in per_group.values())
print(f'\n{len(recs)} people, {total_runs} runs (the study expects 87 and 520)')
print(f'channels: {list(next(iter(chans))) if len(chans) == 1 else chans}')
print('checks:', 'all passed' if not problems else problems[:10])

(OUT / 'manifest.json').write_text(json.dumps(dict(
    source='Dreyer 2023 (MOABB Dreyer2023, neuralfetch Dreyer2023Large)', licence='CC-BY-4.0',
    preprocessing=dict(sfreq=SFREQ, notch=NOTCH, bandpass=BANDPASS, scaler='RobustScaler per run',
                       baseline=None, clamp=CLAMP, stored_dtype='float16'),
    windows=dict(start=START, duration=DURATION, labels={'left_hand': 0, 'right_hand': 1}),
    split=dict(test=TEST_SUBJECTS, val=sorted(val), val_note='ours: 20% of non-test people, seed 33'),
    per_group=per_group, recordings=recs, failed={str(k): v for k, v in failed.items()}), indent=1))
size = sum(f.stat().st_size for f in WIN.glob('*.npz')) / 1e6
print(f'\nmanifest -> {OUT / "manifest.json"}   windows on Drive: {size:.0f} MB')
ready = len(recs) == 87 and total_runs == 520 and not failed and not problems
print('READY for track2_02' if ready else 'NOT READY: see the counts and checks above')